In [536]:
import pandas as pd
import regex as re
import numpy as np

In [537]:
data = pd.read_csv("data/processed_translation_data_2.csv")

In [538]:
data["author"] = data["author"].str.split(':')
data = data.explode("author")
data["author"] = data["author"].str.strip()

In [539]:
correct_colindex = ["author", "translator", "editor", "fore_afterword_author", 
                    "title", "title_original", "genre", "source_language", "source_literature",
                   "publisher", "publication_location", "publication_year", "physical_description",
                   "series", "publication_type", "target_audience", "edition", "n_pages", "content",
                   "issue", "notes"]

In [540]:
for col in data.columns:
    if col not in correct_colindex:
        print(f"{col} is not in data.columns")

Unnamed: 0 is not in data.columns
author_of_translated_book_under_review_to_be_removed is not in data.columns
id is not in data.columns


In [541]:
data = data.drop(["Unnamed: 0", "id", "author_of_translated_book_under_review_to_be_removed"], axis=1)

In [542]:
data = data.loc[:, correct_colindex]

In [543]:
#data.head(30).to_csv("data/demo-30.csv", index=False)

In [544]:
# Made with the help of ChatGPT (unfortunately)
def split_name_year(series):
    """Split into name and year parts."""
    return series.str.split(r"(\d.*)", n=1, expand=True)[[0,1]]

def clean_text(series):
    return series.str.strip(" \t,")

def split_years(series):
    return series.str.split(r"-", n=1, expand=True)

def clean_year(series):
    return (
        series
        .astype(str)
        .str.replace(r"[^\d\-]", "", regex=True)
        .str.strip()
        .replace("", np.nan)
    )


In [545]:
# Made with the help of ChatGPT (unfortunately)
data[["author_name", "author_year"]] = split_name_year(data["author"])
data[["translator_name", "translator_year"]] = split_name_year(data["translator"])

data["author_name"] = clean_text(data["author_name"])
data["translator_name"] = clean_text(data["translator_name"])

# Split birth/death years
data[["birth_year", "death_year"]] = split_years(data["author_year"])

In [546]:
# Made with the help of ChatGPT (unfortunately)
# --- Build persons dataframe ---
persons = pd.concat([data["author"], data["translator"]]).dropna().drop_duplicates()

persons_df = pd.DataFrame()
persons_df[["name", "year"]] = split_name_year(persons)
persons_df = persons_df.reset_index(drop=True)

# Fix known problematic cases
replacements = {
    "497/496-406/405 eKr: Demokritos u 460- u 370 eKr": "497-406",
    "497/496-406/405 eKr": "497-406",
    "1213(1219)-1292": "1213-1292"
}
persons_df["year"] = persons_df["year"].replace(replacements)

# Split birth/death years using broader separators
persons_df[["birth_year", "death_year"]] = persons_df["year"].str.split(
    r"[\/\-\.—–\(\)]", n=1, expand=True
)

# Clean values
persons_df["birth_year"] = clean_year(persons_df["birth_year"])
persons_df["death_year"] = clean_year(persons_df["death_year"])

# Convert to numeric
persons_df["birth_year"] = pd.to_numeric(persons_df["birth_year"], errors="coerce")
persons_df["death_year"] = pd.to_numeric(persons_df["death_year"], errors="coerce")

# Drop intermediate column
persons_df = persons_df.drop(columns=["year"])

# --- Done ---
print(persons_df.head())

                         name  birth_year  death_year
0                  Zunk, Paul         NaN         NaN
1          Žukovski, Vassili       1783.0      1852.0
2                Žukov, V. V.         NaN         NaN
3          Zshokke, Heinrich       1771.0      1848.0
4  Zschokke, Heinrich Daniel       1771.0      1848.0


In [553]:
persons_df.name = persons_df.name.str.strip()

In [555]:
persons_df.to_csv("data/persons.csv", index=False, quotechar="")

TypeError: "quotechar" must be a unicode character or None, not a string of length 0